In [1]:
import re

def clean_text(raw_text):
    # Remove figure/table captions
    text = re.sub(r'(Figure|Table)\s*\d+.*?\n', '', raw_text)
    # Remove references section
    text = re.sub(r'References[\s\S]*', '', text)
    # Remove page numbers
    text = re.sub(r'\n?\s*\d+\s*\n', '\n', text)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


In [2]:
cleaned_text = clean_text("book1.pdf")

In [3]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as pdf:
        for page in pdf:
            text += page.get_text("text")
    return text

pdf_text = extract_text_from_pdf(cleaned_text)
print(pdf_text[:1000])  # preview


INTRODUCTION 
TO 
LINEAR ALGEBRA 
Third Edition 
GILBERT STRANG 
Massachusetts Institute of Technology 
WELLES LEY·(AMBRI DGE PRESS 
Box 812060 Wellesley MA 02482 
introduction to Linear Algebra, 3rd Edlllon 
Copyrigh~ C>2OO3 by GUbe" Strang 
All righlS ~seryed. No pan of this work may be "'produced or storw or transmitted by any 
moans. including photocopying. without wrinen permin ion from WeJJesley--CllIIIbridge fUss. 
Translation in any language is strictly prohibited 
authorized tBllSlations are amoged. 
Printed in the United States of Amoriea 
ISBN 0·9614088-9-8 
QAIS4.S78 
2003 
5 12'.5 
93-14092 
Other tulS from Wt lles~y-Cambrldge Press 
Wa"elels and Filter Banks, Gilben Strang and Truong Ngu)·cn. 
ISBN 0-%14088·7-1. 
Llnfllr Algebra, Geodny, and GPS, Gilben Strang and Kai Borre. 
ISBN 0-%14088-6-3. 
Introduction 10Applied Mathemali .... Gilbe" Strang. 
ISBN 0-%14088~-4 . 
An Analysis orthe Flnltt Element Method. Gilben Strang and George FiJt. 
ISBN 0·%14088·8-X. 
Calculus. Gi

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
)

chunks = splitter.split_text(pdf_text)
print(f"Total chunks: {len(chunks)}")


Total chunks: 1592


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en")

vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embedding,  # ✅ embedding object, not a function
    collection_name="spectral_analysis_local",
    persist_directory="./chroma_local"
)


In [9]:
#query = "Explain the relationship between spectral analysis and Fourier transform."
query = "Explain the correlation between matrix diagonalization and its eigenvalues and eigenvectors."
results = vectorstore.similarity_search(query, k=3)

context = "\n\n".join([r.page_content for r in results])
print("Context:\n", context[:800])


Context:
 259 
6 
Eigenvalues and Eigenvectors 
274 
6.1 
Introduction to Eigenvalues 
27. 
6.2 
Diagonalizing a Manu 
288 
63 
Applications to Differential Equations 
304 
6A 
Symmetric Matrices 
318 
iii 
6.S 
Positive Definite Matrices 
6.6 
Similar Matrices 
6.7 
Singular Value Decomposition (SVD) 
7 
linear Transformations 
7.1 
The Idea of a Linear Transformation 
7.2 
The Matrix of a Linear Transfonnation 
7.3 
Change of Basis 
7.4 
Diagona.lization and the Pseudoinverse 
8 
Applications 
8.1 
Matrices in Engineering 
8.2 
Graphs and Networks 
8.3 
Markov Matrices and Economic Models 
8.4 
Linear Programming 
8.S 
Fourier Series: Linear Algebra for Functions 
8.6 
Computer Graphics 
9 
Numerical linear Algebra 
9.1 
Gaussian Elimination in Practice 
9.2 
Norms and Condition Numbers 
9.3

259 


In [ ]:
# in the BASH: ollama pull llama3 (pull the model)

In [10]:
import subprocess

#prompt = f"""
#You are an expert in signal processing.
#Using the context below, answer the question clearly.

#Context:
#{context}

#Question: {query}
#"""

prompt = f"""
You are and expert in matrix diagonalization, eigenvalues and eigenvectors.
Using the context below, answer the question clearly.

Context:
{context}

Question: {query}
"""

# Send prompt to local Llama 3 model
result = subprocess.run(
    ["ollama", "run", "llama3"],
    input=prompt.encode("utf-8"),
    capture_output=True
)

print("\n--- Llama 3 Answer ---\n")
print(result.stdout.decode("utf-8"))



--- Llama 3 Answer ---

In the given context, it seems that you are discussing the diagonalization of a matrix B using singular value decomposition (SVD). The equation (4) implies that when the matrix 8 is diagonalized, one of its eigenvalues goes to zero.

The example provided illustrates the importance of eigenvalues and eigenvectors in matrix diagonalization. It highlights how the off-diagonal entries of the matrix B' set the speed of convergence, indicating that the eigenvalues (or singular values) play a crucial role in this process.

The key takeaways from this explanation are:

1. **Eigenvalues influence diagonalization**: The values on the diagonal of the matrix after diagonalization (B') depend on the original matrix's eigenvalues (or singular values).
2. **Eigenvectors matter too**: When diagonalizing a matrix, it is essential to have independent eigenvectors as a basis. Without this, diagonalization becomes impossible.
3. **Eigenvalue convergence**: The example shows that w